# LoRA-corrected teacher eval: MBPP-TS (full 390-problem set)

Part of plan.md step 3.3 -- the second gating benchmark, run only against
the LoRA-corrected teacher (`sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora`). Not run against the
full-parameter corrected teacher: that checkpoint was trained before the
3.4 data-quality fix (truncated samples, `node_modules`/`.d.ts`
contamination), so its MBPP-TS score wouldn't be a fair data point.

Same suite/harness as `11b_teacher_corrected_lora_humaneval_ts.ipynb`,
same export-stripping fix (`bonsai.eval.register`), `batch_size=8`, and
`--resume` (checkpoints after each batch via `save_every_k_tasks=1`).

Executed 2026-09-26/27 on a DigitalOcean 1x H100 (80GB): ~9h53min
generation + ~3min execution/scoring for 390 problems.

In [1]:
from pathlib import Path

from bonsai.config import load_config

config = load_config("../configs/qwen3-30b-a3b.toml")
model_repo_id = "sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora"
run_dir = Path("../runs/20260926T143533Z-572b8e-teacher-corrected-lora-eval")

print(f"model: {model_repo_id}")
print(f"run dir: {run_dir}")

model: sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora
run dir: ../runs/20260926T143533Z-572b8e-teacher-corrected-lora-eval


In [2]:
import os

os.environ["MODEL_REPO_ID"] = model_repo_id
os.environ["MODEL_REVISION"] = "main"
os.environ["RUN_DIR"] = str(run_dir)

In [3]:
!stdbuf -o0 -e0 accelerate launch ../tools/bigcode_eval_main.py \
    --model "$MODEL_REPO_ID" --revision "$MODEL_REVISION" --precision bf16 \
    --tasks multiple-mbpp-ts \
    --max_length_generation 512 --batch_size 8 \
    --allow_code_execution --save_generations --resume \
    --metric_output_path "$RUN_DIR/mbpp-ts.json" \
    --save_generations_path "$RUN_DIR/mbpp-ts-generations.json" \
    2>&1 | tee /tmp/live_mbpp_ts_lora_full.log

assert _exit_code == 0, "accelerate launch failed for multiple-mbpp-ts"  # noqa: F821 -- set by the ! shell magic above at runtime

Selected Tasks: ['multiple-mbpp-ts']
resume: no checkpoint at runs/20260926T143533Z-572b8e-teacher-corrected-lora-eval/mbpp-ts-generations_multiple-mbpp-ts_intermediate.json, starting from scratch for multiple-mbpp-ts
Loading model in bf16
number of problems for this task is 390
... generation (390/390, ~9h53min, checkpoints every batch) ...
Evaluating generations...
... execution/scoring (390/390, ~3min) ...
{
  "multiple-mbpp-ts": {
    "pass@1": 0.6076923076923076
  }
}


In [4]:
import json

mbpp_ts = json.loads((run_dir / "mbpp-ts.json").read_text())
print("mbpp-ts (LoRA-corrected teacher, full 390-problem set):", mbpp_ts)

mbpp-ts (LoRA-corrected teacher, full 390-problem set): {'multiple-mbpp-ts': {'pass@1': 0.6076923076923076}, 'config': {'model': 'sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora', 'tasks': 'multiple-mbpp-ts', 'batch_size': 8, 'resume': True, 'save_every_k_tasks': 1, 'metric_output_path': 'runs/20260926T143533Z-572b8e-teacher-corrected-lora-eval/mbpp-ts.json'}}


## Comparison

| | HumanEval-TS pass@1 | MBPP-TS pass@1 |
|---|---|---|
| base model (unpruned, uncorrected) | 0.6708 | 0.7333 |
| LoRA-corrected teacher | 0.5723 (0.5849 console.log-stripped) | **0.6077** |

LoRA correction is **~12.6pp below** the base on MBPP-TS (vs **~9.8pp** on HumanEval-TS raw). Still below base on both primary benchmarks; general-knowledge suite unchanged (see `11c`).